In [1]:
# ruff: noqa
import sys, os
sys.path.append(os.path.abspath("./../feedback-grape"))
sys.path.append(os.path.abspath("./../"))

# ruff: noqa
from feedback_grape.fgrape import optimize_pulse
from helpers import (
    init_fgrape_protocol,
    test_implementations,
    generate_superposition_state,
    experiment_param_formats,
    generate_povm,
    generate_hermitian,
)
from library.utils.FgResult_to_dict import FgResult_to_dict
import json, jax, time
import jax.numpy as jnp

test_implementations()

In [2]:
batch_size = 100
N_qubits = 2
base_dim = 2**N_qubits
N_povm_params = base_dim*(base_dim+1)
key = jax.random.PRNGKey(42)
povm_params_batch = jax.random.uniform(key, (batch_size, N_povm_params), minval=0.0, maxval=2*jnp.pi)

@jax.jit
def f(povm_params_batch):
    def f2(povm_params):
        sum = 0.0
        for _ in range(10):
            povm = generate_povm(+1, params=povm_params, dim=base_dim)
            sum = sum + jnp.abs(jnp.sum(povm))
        return sum
    
    povm_params_batch = jax.vmap(f2)(povm_params_batch)
    return jnp.sum(povm_params_batch)

f(povm_params_batch)

start = time.time()
for _ in range(100):
    f(povm_params_batch)
print("Time taken:", time.time() - start)

start = time.time()
for _ in range(100):
    value, grad = jax.value_and_grad(f)(povm_params_batch)
print("Time taken:", time.time() - start)

Time taken: 0.09705829620361328
Time taken: 9.47058916091919


In [3]:
N = 10000
batch_size = 1
arr = jnp.arange(N, dtype=jnp.float64)
arr_batch = jnp.tile(arr, (batch_size, 1))

@jax.jit
def f(arr_batch):
    def f2(arr):
        sum = arr[0:N//2] + arr[N//2:N]
        for i in range(100):
            sum = sum + arr[0:N//2]
            sum = sum + arr[N//2:N]
        return sum
    
    arr_batch = jax.vmap(f2)(arr_batch)

    return jnp.sum(arr_batch)

f(arr_batch)

start = time.time()
for _ in range(100):
    f(arr_batch)
print("Time taken:", time.time() - start)

start = time.time()
for _ in range(100):
    value, grad = jax.value_and_grad(f)(arr_batch)
print("Time taken:", time.time() - start)

Time taken: 0.004209041595458984
Time taken: 0.9591355323791504


In [ ]:
batch_size = 100
N_qubits = 2
base_dim = 2**N_qubits
N_params = base_dim*base_dim
key = jax.random.PRNGKey(42)
params_batch = jax.random.uniform(key, (batch_size, N_params), minval=0.0, maxval=2*jnp.pi)

@jax.jit
def f(params_batch):
    def f2(params):
        sum = 0.0
        for _ in range(10):
            hermitian = generate_hermitian(params, dim=base_dim)
            sum = sum + jnp.abs(jnp.sum(hermitian))
        return sum
    
    params_batch = jax.vmap(f2)(params_batch)
    return jnp.sum(params_batch)

f(params_batch)

start = time.time()
for _ in range(100):
    f(params_batch)
print("Time taken:", time.time() - start)

start = time.time()
for _ in range(100):
    value, grad = jax.value_and_grad(f)(params_batch)
print("Time taken:", time.time() - start)

# Time taken: 0.0042765140533447266
# Time taken: 0.6531970500946045

Time taken: 0.0042765140533447266
Time taken: 0.6531970500946045
